In [ ]:
# Optional: install deps in a fresh environment
# !pip install -q torch transformers accelerate bitsandbytes huggingface_hub

In [ ]:
import os
from huggingface_hub import login

hf_token = os.getenv("HF_TOKEN")
if hf_token:
    login(token=hf_token)
else:
    print("HF_TOKEN not set. If the model is gated, set HF_TOKEN and rerun this cell.")

## Load Dataset


In [ ]:
import json
import os
from pathlib import Path

In [ ]:
def find_repo_root(start: Path) -> Path:
    for p in [start] + list(start.parents):
        if (p / "README.md").exists() and (p / "Dataset").exists():
            return p
    return start

PROJECT_ROOT = find_repo_root(Path.cwd())
TOP_K = 30  # must match the file name produced by the pruning notebook
DATASET_PATH = PROJECT_ROOT / "outputs" / "dataset" / "mkp" / f"llm_dataset_v2_mkp_top_{TOP_K}.json"

def load_dataset(path: Path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

## Model Pipeline (LLaMa 3.2)

In [ ]:
import transformers
import torch

model_id = "meta-llama/Llama-3.2-3B-Instruct"

tokenizer = transformers.AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.pad_token_id = tokenizer.eos_token_id
tokenizer.padding_side = "left"

pipeline = transformers.pipeline(
    "text-generation",
    model=model_id,
    tokenizer=tokenizer,
    torch_dtype="auto",
    device_map="auto",
    max_new_tokens=512,
    do_sample=False,
    pad_token_id=tokenizer.pad_token_id,
)

pipeline.model.config.pad_token_id = tokenizer.pad_token_id

## Start Inference

In [ ]:
from tqdm import tqdm

BATCH_SIZE = 4
MAX_SAMPLES = 100  # set None to run all

In [ ]:
if not DATASET_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found: {DATASET_PATH}. Run the pruning notebook to generate it."
    )

dataset = load_dataset(DATASET_PATH)
print("Loaded", len(dataset), "rows from", DATASET_PATH)

In [ ]:
response_list = []

total_prompt_tokens = 0
total_output_tokens = 0
token_stats = []

In [ ]:
print(dataset[0]["messages"][0:2])

In [ ]:
subset = dataset if MAX_SAMPLES is None else dataset[:MAX_SAMPLES]

num_batches = (len(subset) + BATCH_SIZE - 1) // BATCH_SIZE

for start in tqdm(range(0, len(subset), BATCH_SIZE), total=num_batches, desc="Generating SQL"):
    batch_data = subset[start:start+BATCH_SIZE]
    batch_prompts = [
        tokenizer.apply_chat_template(data["messages"][0:2], tokenize=False, add_generation_prompt=True)
        for data in batch_data
    ]

    # Count prompt tokens for the batch
    batch_prompt_enc = tokenizer(batch_prompts, add_special_tokens=False)
    prompt_lens = [len(ids) for ids in batch_prompt_enc["input_ids"]]

    outputs = pipeline(
        batch_prompts,
        max_new_tokens=512,
        do_sample=False,
        return_full_text=False,
        batch_size=BATCH_SIZE,
    )

    for out, data, prompt_len in zip(outputs, batch_data, prompt_lens):
        generated = out["generated_text"]
        response = generated.strip()
        sql = response + "\t----- bird -----\t" + data["db_id"]
        response_list.append(sql)
        tqdm.write(sql)

        output_len = len(tokenizer(response, add_special_tokens=False)["input_ids"])
        total_prompt_tokens += prompt_len
        total_output_tokens += output_len
        token_stats.append({
            "question_id": data.get("question_id"),
            "db_id": data.get("db_id"),
            "prompt_tokens": prompt_len,
            "output_tokens": output_len,
            "total_tokens": prompt_len + output_len,
        })

# Summary
num_questions = len(subset)
all_tokens = total_prompt_tokens + total_output_tokens
avg_tokens = all_tokens / num_questions if num_questions else 0
avg_prompt_tokens = total_prompt_tokens / num_questions if num_questions else 0
avg_output_tokens = total_output_tokens / num_questions if num_questions else 0

print("Total questions:", num_questions)
print("Total prompt tokens:", total_prompt_tokens)
print("Total output tokens:", total_output_tokens)
print("Total tokens:", all_tokens)
print("Avg tokens/question:", avg_tokens)
print("Avg prompt tokens/question:", avg_prompt_tokens)
print("Avg output tokens/question:", avg_output_tokens)

In [ ]:
print(response_list[0:10])

## Save Results

In [ ]:
import os

In [ ]:
def new_directory(path):
    Path(path).mkdir(parents=True, exist_ok=True)

In [ ]:
def generate_sql_file(sql_lst, output_path=None):
    result = {i: sql for i, sql in enumerate(sql_lst)}

    if output_path:
        output_path = Path(output_path)
        directory_path = output_path.parent
        new_directory(directory_path)
        with output_path.open("w", encoding="utf-8") as f:
            json.dump(result, f, indent=4, ensure_ascii=False)

    return result

In [ ]:
output_path = PROJECT_ROOT / "outputs" / "predictions" / f"llama_3.2_mkp_top_{TOP_K}.json"
generate_sql_file(response_list, output_path)
print("Saved:", output_path)